In [ ]:
import cv2
import os
from pathlib import Path

IMAGES_DIR = "individual_robots"  # folder with original images
LABELS_DIR = "runs/detect/predict/labels" # folder with YOLO labels
OUTPUT_DIR = "robot_numbers" # folder to save cropped images
CLASS_ID = 1  # class ID to filter (e.g., 1 for robots)

os.makedirs(OUTPUT_DIR, exist_ok=True)

def yolo_to_pixel(img_w, img_h, x, y, w, h):
    xc, yc = x * img_w, y * img_h
    bw, bh = w * img_w, h * img_h
    x1 = int(xc - bw / 2)
    y1 = int(yc - bh / 2)
    x2 = int(xc + bw / 2)
    y2 = int(yc + bh / 2)
    return max(0, x1), max(0, y1), min(img_w, x2), min(img_h, y2)

count = 0

for img_path in Path(IMAGES_DIR).glob("*.*"):
    label_path = Path(LABELS_DIR, img_path.stem + ".txt")
    if not label_path.exists():
        continue

    image = cv2.imread(str(img_path))
    if image is None:
        continue

    h, w, _ = image.shape

    with open(label_path) as f:
        for i, line in enumerate(f):
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id, x, y, bw, bh = map(float, parts)

            # KEEP ONLY CHOSEN CLASS
            if int(class_id) != CLASS_ID:
                continue

            x1, y1, x2, y2 = yolo_to_pixel(w, h, x, y, bw, bh)
            crop = image[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            out_name = f"{img_path.stem}_obj{i}_class1.jpg"
            cv2.imwrite(os.path.join(OUTPUT_DIR, out_name), crop)
            count += 1

print(f"Saved {count} crops of class 1")

✅ Saved 284 crops of class 1
